# Tennessee Eastman Process — Drift-Erkennung

Wendet die vier Drift-Detektoren KSWIN, ADWIN, DDM und EDDM (`drift_detection.py`) auf den normierten Modellfehler des Fehlerfallstroms (IDV 29) aus *tep_model* an; als Ground Truth dient das aus `drift_functions.c` rekonstruierte Driftsignal. Da die Simulationsläufe kürzer als die Sägezahn-Periode (800 h) sind, treten keine Sudden Drifts auf; Tuning und Kennzahlen beziehen sich auf den Driftbeginn je Lauf.

In [ ]:
import os
from pathlib import Path

work_dir = os.getcwd()
DEFAULT_BASE_DIR = os.path.normpath(os.path.join(work_dir, "..", ".."))

base_dir = Path(os.environ.get("BASE_DIR", DEFAULT_BASE_DIR)).resolve()
data_dir = base_dir / "data" / "tep"
model_dir = base_dir / "models"
plot_dir = base_dir / "plots"
results_dir = base_dir / "results"

for _d in (data_dir, model_dir, plot_dir, results_dir):
    os.makedirs(_d, exist_ok=True)
    
FORCE_RECOMPUTE = False  # True = alle Cache-Stufen neu rechnen (Tuning + nachgelagerte Laeufe)
IS_FINAL = True  # True: schreibt zusaetzlich den stabilen, eingebundenen Stand; False: nur Archiv

In [ ]:
import numpy as np

In [ ]:
# --- Konstanten ---
SEED = 1
RUN_ID = None
ERROR_VARIANT = None   # None -> neuester Fehlerfall; sonst z. B. {"idv": 29, "amp": 1.0}
SAMPLES_PER_H = 20
DRIFT_PERIOD, DRIFT_GAMMA, DRIFT_SEED = 800.0, 0.2, 42
DETECTORS = ["KSWIN", "ADWIN", "DDM", "EDDM"]
TOLERANCE = 2000   # Samples (100 h)
N_TRIALS  = {
    "KSWIN": 40,
    "ADWIN": 40,
    "DDM": 200,
    "EDDM": 200
}
COOLDOWN  = 1000   # Samples (50 h)

rng = np.random.default_rng(SEED)

Daten laden: gescorte Test- und Fehlerfalldaten aus dem Modell-Notebook

In [ ]:
from src.utils import run_registry as rr
ledger = rr.run_ledger(work_dir)
data_store = rr.tep_data_store(data_dir)
model_store = rr.tep_model_store(model_dir)
plot_store = rr.tep_plot_store(plot_dir, ledger=ledger)
res_store = rr.tep_results_store(results_dir, ledger=ledger)

_mp = model_store.latest("model")
if _mp is None:
    raise FileNotFoundError("Kein model-Artefakt. Bitte tep_model.ipynb ausfuehren.")
model_id = model_store.parse_id(_mp, "model") if RUN_ID is None else RUN_ID

test_scored = data_store.load("data_scored_production", run_id=model_id, rename=True)
model_cfg = rr.RunConfig.from_document(test_scored.attrs.get("config", test_scored.attrs))

error_scored, scored_error_cfg = rr.load_section(
    data_store, "data_scored_error", model_cfg, "error", ERROR_VARIANT)
error_variant = scored_error_cfg.sections["error"]
ledger.bind("detection", parents={"model": scored_error_cfg.id})
print(f"model_id = {model_id} | error_run_id = {scored_error_cfg.id} "
      f"(IDV {error_variant['idv']}, amp {error_variant['amp']})")

## Fehlerstrom normieren

Der Modellfehler des Fehlerfallstroms wird mit Mittelwert und Streuung des drift-freien Testfehlers normiert, konsistent zur Normierung im synthetischen Fall.

In [ ]:
_err_free = test_scored["model_error"].to_numpy(dtype=float)
NORM_MEAN = float(np.nanmean(_err_free))
NORM_STD = float(np.nanstd(_err_free)) or 1.0

error_stream = np.nan_to_num((error_scored["model_error"].to_numpy(dtype=float) - NORM_MEAN) / NORM_STD)
reset_pos = np.flatnonzero(error_scored["reset"].to_numpy())
N = len(error_stream)
print(f"Fehlerstrom: {N} Zeitschritte ({N / SAMPLES_PER_H:.0f} h), {len(reset_pos) + 1} Laeufe | "
      f"Normierung: mean={NORM_MEAN:.4g}, std={NORM_STD:.4g}")

## Driftsignal rekonstruieren

Python-Port von `drift_function_sawtooth.c` (bitidentisch, splitmix64) mit den Parametern aus `temexd_mod.c` (Periode 800 h, $\gamma = 0{,}2$, Seed 42), ausgewertet an der Simulationszeit, die je Lauf bei 0 startet.

In [ ]:
_M64 = (1 << 64) - 1
CD_MID, CD_HALF = 0.0, 0.5   # Spiegel der #defines in drift_function_sawtooth.c

def _drift_rand(seed, idx):
    z = (seed + idx * 0x9E3779B97F4A7C15) & _M64
    z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & _M64
    z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & _M64
    z = z ^ (z >> 31)
    return (z >> 11) / 9007199254740992.0

def _drift_uniform(seed, idx, lo, hi):
    return lo + (hi - lo) * _drift_rand(seed, idx)

def _drift_boundary(k, period, gamma, seed):
    if k <= 0:
        return 0.0
    return k * period + (2.0 * _drift_rand(seed, 1000 + k) - 1.0) * gamma * 0.5 * period

def drift_signal(t, period=DRIFT_PERIOD, gamma=DRIFT_GAMMA, seed=DRIFT_SEED):
    if t <= 0.0 or period <= 0.0:
        return 0.0
    k = int(np.floor(t / period))
    while k > 0 and t < _drift_boundary(k, period, gamma, seed):
        k -= 1
    while t >= _drift_boundary(k + 1, period, gamma, seed):
        k += 1
    t0 = _drift_boundary(k, period, gamma, seed)
    t1 = _drift_boundary(k + 1, period, gamma, seed)
    c0 = _drift_uniform(seed, 2000 + k, CD_MID + CD_HALF / 4.0, CD_MID + CD_HALF)
    c1 = _drift_uniform(seed, 3000 + k, CD_MID - CD_HALF, CD_MID - CD_HALF / 4.0)
    return c0 + (c1 - c0) * (t - t0) / (t1 - t0)

sim_time = error_scored["time"].to_numpy(dtype=float)
cd_signal = error_variant["amp"] * np.array([drift_signal(t) for t in sim_time])
print(f"Driftsignal: min={cd_signal.min():.3f}, max={cd_signal.max():.3f}, "
      f"erste Segmentgrenze bei {_drift_boundary(1, DRIFT_PERIOD, DRIFT_GAMMA, DRIFT_SEED):.0f} h")

## Ground Truth: Driftbeginn je Lauf

Als Ereignis dient das Überschreiten der Aktivitätsschwelle (10 % der maximalen Driftamplitude im Strom) je Lauf; Läufe, die kürzer als die Anlaufzeit sind, liefern kein Ereignis.

In [ ]:
import drift_detection as dd

drift_active = dd.drift_active_mask(cd_signal, rel_floor=0.1)
onset_idx = (np.flatnonzero(np.diff(drift_active.astype(int)) == 1) + 1).tolist()
if drift_active[0]:
    onset_idx = [0] + onset_idx

_jumps_in_run = np.flatnonzero((np.diff(cd_signal) > 0.1 * np.nanmax(np.abs(cd_signal)))
                              & (np.diff(sim_time) > 0))
print(f"{len(onset_idx)} Driftbeginn-Ereignisse bei "
      + ", ".join(f"{i / SAMPLES_PER_H:.0f} h" for i in onset_idx))
print(f"Sudden Drifts innerhalb der Laeufe: {len(_jumps_in_run)}")

## Parameter-Tuning (Optuna)

In [ ]:
from types import SimpleNamespace
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

det_cfg = model_cfg.compose(error=error_variant,
                      detector={"events": "onset", "detectors": DETECTORS,
                      "n_trials": N_TRIALS, "tolerance": TOLERANCE,
                      "weights": [3.0, 0.3, 0.05], "downsample": 1})

_tuning_loaded_from_cache = model_store.exists("detect_tuning", det_cfg, rekey=True) and not FORCE_RECOMPUTE
if _tuning_loaded_from_cache:
    _tuning = model_store.load("detect_tuning", det_cfg, rename=True, rekey=True)
else:
    _tuning = {"best_params": {}, "user_attrs": {}, "best_value": {}}

_retuned = False
studies, best_params = {}, {}

for _name in DETECTORS:
    if _name in _tuning["best_params"]:
        best_params[_name] = _tuning["best_params"][_name]
        studies[_name] = SimpleNamespace(
            best_value=_tuning["best_value"][_name],
            best_trial=SimpleNamespace(user_attrs=_tuning["user_attrs"][_name]))
        print(f"{_name:6s} aus Cache geladen")
        continue
    studies[_name] = dd.tune_detector(
        _name, error_stream, onset_idx, drift_active,
        tolerance=TOLERANCE, n_trials=N_TRIALS[_name], downsample=1, seed=SEED,
        weights=(3.0, 0.3, 0.05))
    best_params[_name] = studies[_name].best_params
    _tuning["best_params"][_name] = best_params[_name]
    _tuning["user_attrs"][_name] = dict(studies[_name].best_trial.user_attrs)
    _tuning["best_value"][_name] = float(studies[_name].best_value)
    _retuned = True

if _retuned:
    model_store.save(_tuning, "detect_tuning", det_cfg)

for _name in DETECTORS:
    a = studies[_name].best_trial.user_attrs
    print(f"{_name:6s} score={studies[_name].best_value:.3f} | recall={a['recall']:.2f} "
          f"delay={a['mean_delay'] / SAMPLES_PER_H:.1f} h false={a['n_false']:3d} -> {best_params[_name]}")

In [ ]:
det_run = det_cfg.compose(cooldown=COOLDOWN)
_detections_loaded_from_cache = model_store.exists("detections", det_run, rekey=True) and not FORCE_RECOMPUTE and not _retuned
if _detections_loaded_from_cache:
    _det_cache = model_store.load("detections", det_run, rename=True, rekey=True)
    tuned = _det_cache["tuned"]
    for _name in DETECTORS:
        print(f"{_name:6s}: {len(tuned[_name][0]):3d} Drift(s) (getunt)")
else:
    tuned = {}
    for _name, _p in best_params.items():
        _p = dict(_p)
        _et = _p.pop("err_threshold", 0.5)
        _det = dd.build_detector(_name, **_p)
        tuned[_name] = dd.run_detector(_name, _det, error_stream, err_threshold=_et)
        print(f"{_name:6s}: {len(tuned[_name][0]):3d} Drift(s) (getunt)")

## Vergleich der getunten Detektoren

Cooldown unterdrückt Wiederholdetektionen der fensterbasierten Detektoren KSWIN und ADWIN während des andauernden inkrementellen Drifts.

In [ ]:
if _detections_loaded_from_cache:
    norepeat = _det_cache["norepeat"]
    for _name in ["KSWIN", "ADWIN"]:
        print(f"{_name:6s}: {len(norepeat[_name][0]):3d} Drift(s) (cooldown={COOLDOWN}, aus Cache)")
else:
    norepeat = {}
    print(f"{'Detektor':8s}{'cooldown':>10s}{'Drifts':>9s}{'recall':>8s}{'false_stabil':>14s}")
    for _name in ["KSWIN", "ADWIN"]:
        _p = dict(best_params[_name]); _p.pop("err_threshold", None)
        for _cd in [0, 500, 1000, 2000]:
            _det = dd.build_detector(_name, **_p)
            _res = dd.run_detector(_name, _det, error_stream, cooldown=_cd)
            _drifts = _res[0]
            _sc = dd.score_detections(_drifts, onset_idx, drift_active,
                                      tolerance=TOLERANCE, n=N)
            print(f"{_name:8s}{_cd:10d}{len(_drifts):9d}{_sc['recall']:8.2f}{_sc['n_false']:14d}")
            if _cd == COOLDOWN:
                norepeat[_name] = _res
    model_store.save({"tuned": tuned, "norepeat": norepeat}, "detections", det_run)

Die folgenden Abbildungen zeigen je Detektor den normierten Modellfehler, das rekonstruierte Driftsignal (an den Resets unterbrochen) und die Detektionszeitpunkte (KSWIN/ADWIN mit Cooldown, DDM/EDDM ohne).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from src.utils import thesis_style as ts


t_h = np.arange(N) / SAMPLES_PER_H
ALPHA, S, FRAC = 0.5, 6, 0.2
C_ERR, C_DRIFT, C_DET = ts.C["drift_afflicted"], ts.C["drift_signal"], ts.C["detection"]
rng_plot = np.random.default_rng(SEED)

detections = {"KSWIN": norepeat["KSWIN"][0], "ADWIN": norepeat["ADWIN"][0],
              "DDM": tuned["DDM"][0], "EDDM": tuned["EDDM"][0]}

def align_zero(ax_left, ax_right):
    """0 beider Ordinaten auf gleiche Hoehe (wie im Modell-Notebook)."""
    ax_left.figure.canvas.draw()
    lims = [ax_left.get_ylim(), ax_right.get_ylim()]
    fracs = [(-lo / (hi - lo) if hi != lo else 0.5) for lo, hi in lims]
    r = min(max(max(fracs), 1e-3), 1 - 1e-3)
    for ax, (lo, hi) in zip((ax_left, ax_right), lims):
        span = max(hi / (1 - r), -lo / r)
        ax.set_ylim(-r * span, (1 - r) * span)

cut = reset_pos
for j, (name, drifts) in enumerate(detections.items()):
    fig, ax = plt.subplots(figsize=(ts.fig_width(), ts.fig_width(0.33)))

    m = rng_plot.random(N) < FRAC
    ax.scatter(t_h[m], error_stream[m], color=C_ERR, s=S, alpha=ALPHA,
               marker=ts.MODEL_MARKER, zorder=3, rasterized=True)

    ax2 = ax.twinx()
    for a, b in zip(np.r_[0, cut], np.r_[cut, N]):
        ax2.plot(t_h[a:b], cd_signal[a:b], **ts.line("drift_signal", linewidth=1.5))
    ax2.set_ylabel("Driftsignal $c(t)$")
    ax2.spines["right"].set_visible(True)
    ax2.spines["top"].set_visible(False)

    for d in drifts:
        ax.axvline(t_h[d], zorder=2, **ts.vline("detection", lw=0.9, linestyle="--"))

    ax.set_ylabel("Norm. Fehler")
    ax.margins(x=0)
    ax.grid(True, alpha=0.2)
    if j == len(detections) - 1:
        ax.set_xlabel("Zeit $t$ [h]")

    if j == 0:
        handles = [Line2D([0], [0], marker=ts.MODEL_MARKER, linestyle="", color=C_ERR,
                          markersize=ts.LEGEND_MARKERSIZE, label="Norm. Fehler"),
                   Line2D([0], [0], color=C_DRIFT, label="Driftsignal $c(t)$"),
                   Line2D([0], [0], color=C_DET, linestyle="--", label="Drift erkannt")]
        ax.legend(handles=handles, loc="lower left", frameon=False)

    align_zero(ax, ax2)
    fig.tight_layout()
    plot_store.save_figure(fig, f"detection_{name.lower()}", det_run, final=IS_FINAL, savefig_kwargs={"dpi": 300}, archive_kwargs={"dpi": 300})
    plt.show()

## Ergebnisse speichern

In [ ]:
from src.utils import results_export as rx

def _full_metrics(drifts):
    return dd.score_detections(drifts, onset_idx, drift_active, tolerance=TOLERANCE, n=N)

detector_results = {}
for _name in DETECTORS:
    _ua = studies[_name].best_trial.user_attrs
    _drifts = tuned[_name][0]
    _m = _full_metrics(_drifts)
    _entry = {
        "best_params": best_params[_name],
        "tuning_metrics": {_k: _ua.get(_k) for _k in
            ["recall", "mean_delay", "n_false", "fa_per_true", "n_sudden", "n_detected", "score"]},
        "tuning_score": studies[_name].best_value,
        "full_stream": {"n_detected": _m["n_detected"], "recall": _m["recall"],
                        "mean_delay": _m["mean_delay"], "n_false": _m["n_false"]},
        "drift_indices": list(_drifts),
    }
    if _name in norepeat:
        _dnr = norepeat[_name][0]
        _mnr = _full_metrics(_dnr)
        _entry["norepeat"] = {"cooldown": COOLDOWN, "n_detected": _mnr["n_detected"],
                              "recall": _mnr["recall"], "mean_delay": _mnr["mean_delay"],
                              "n_false": _mnr["n_false"], "drift_indices": list(_dnr)}
    detector_results[_name] = _entry

(rx.ResultDoc()
   .integer("n_samples", N)
   .set("samples_per_hour", SAMPLES_PER_H)
   .integer("n_onset_events", len(onset_idx))
   .set("onset_indices", onset_idx)
   .set("reset_indices", reset_pos)
   .integer("n_active", np.sum(drift_active))
   .set("tolerance", TOLERANCE)
   .set("cooldown", COOLDOWN)
   .set("tuning", {"n_trials": N_TRIALS, "downsample": 1, "weights": [3.0, 0.3, 0.05]})
   .set("detectors", detector_results)
   .save(res_store, "detection", det_run, final=IS_FINAL,
         parents={"model": model_id}))

In [ ]:
from src.utils import latex_export as lx

mx = lx.MacroExport("automatisch erzeugt aus tep_detection.ipynb - nicht manuell editieren")

mx.comment("Parameter (Hash-relevant)")
mx.integer("tepdriftNOnset", len(onset_idx))
mx.num("tepdriftToleranceH", TOLERANCE / SAMPLES_PER_H, 0)
mx.num("tepdriftCooldownH", COOLDOWN / SAMPLES_PER_H, 0)

mx.comment("Ergebnisse (Detektionskennzahlen je Detektor)")
for _name in DETECTORS:
    _e = detector_results[_name]
    _fs = _e["full_stream"]
    mx.num(f"tepdrift{_name}Recall", _fs["recall"], 2)
    mx.num(f"tepdrift{_name}DelayH", _fs["mean_delay"] / SAMPLES_PER_H, 1)
    mx.integer(f"tepdrift{_name}False", _fs["n_false"])
    mx.integer(f"tepdrift{_name}Drifts", _fs["n_detected"])
    if "norepeat" in _e:
        mx.integer(f"tepdrift{_name}DriftsNoRep", _e["norepeat"]["n_detected"])

mx.save(res_store, "detection", det_run, final=IS_FINAL)
